# ABSA Restaurant tiếng Việt — Clean Ablation Training

Notebook này huấn luyện pipeline ABSA trên bản dịch tiếng Việt của dataset Restaurant.

Khác với notebook tiếng Anh, encoder mặc định là `microsoft/mdeberta-v3-base` (multilingual). Quy trình vẫn giữ nguyên: làm sạch overlap, chia validation theo câu, core ablation trên ba seed, chọn bằng robust Macro-F1, refit full train và đánh giá official test.

Trước khi chạy:

1. Upload `absa_rest_vi_ablation_fresh.zip` thành Kaggle Dataset.
2. Gắn Dataset đó vào notebook.
3. Bật GPU và Internet.
4. Chạy toàn bộ cell theo thứ tự.

In [1]:
from pathlib import Path
import os, shutil, zipfile, json, subprocess, sys

print('Python:', sys.version)
print('Kaggle working exists:', Path('/kaggle/working').exists())
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('Không tìm thấy nvidia-smi; kiểm tra lại GPU accelerator.')

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Kaggle working exists: True
Mon Jun 22 19:44:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |          

## 1. Giải nén package và tạo trạng thái clean-start

Cell tự tìm ZIP trong Kaggle Input. Nếu Kaggle đã giải nén Dataset, notebook tìm `train.py` và copy thư mục nguồn.

In [2]:
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
INPUT = Path('/kaggle/input/datasets/banhbaogachcua/nlp-source-vi')
PROJECT = WORK / 'absa_rest_vi_ablation_fresh'

CORE_TUNING = WORK / 'tuning_rest_vi_ablation_core'
CORE_FINAL = WORK / 'outputs_rest_vi_ablation_core'
RESULT_ZIP = WORK / 'absa_rest_vi_results_fresh.zip'

CLEAN_RUN = True
if CLEAN_RUN:
    for path in [PROJECT, CORE_TUNING, CORE_FINAL, RESULT_ZIP]:
        if path.exists():
            print('Removing old:', path)
            shutil.rmtree(path) if path.is_dir() else path.unlink()

if not INPUT.exists():
    raise FileNotFoundError(f'Không tìm thấy Kaggle Input: {INPUT}')

zip_candidates = list(INPUT.rglob('absa_rest_vi_ablation_fresh.zip'))
if not zip_candidates:
    zip_candidates = list(INPUT.rglob('*rest*vi*ablation*.zip'))

if zip_candidates:
    zip_path = zip_candidates[0]
    print('Extracting ZIP:', zip_path)
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(PROJECT)
else:
    candidates = [p for p in INPUT.rglob('train.py') if (p.parent / 'dataset/rest_vi').exists()]
    if not candidates:
        visible = [str(p.relative_to(INPUT)) for p in INPUT.rglob('*') if p.is_file()][:50]
        raise FileNotFoundError('Không tìm thấy package Restaurant VI. Files: ' + str(visible))
    source_dir = candidates[0].parent
    print('Copying extracted source:', source_dir)
    shutil.copytree(source_dir, PROJECT, dirs_exist_ok=True)

if not (PROJECT / 'train.py').exists():
    candidates = list(PROJECT.rglob('train.py'))
    if not candidates:
        raise FileNotFoundError('Đã giải nén nhưng không tìm thấy train.py')
    PROJECT = candidates[0].parent

print('PROJECT =', PROJECT)
print('Files:', sorted(p.name for p in PROJECT.iterdir()))

Copying extracted source: /kaggle/input/datasets/banhbaogachcua/nlp-source-vi
PROJECT = /kaggle/working/absa_rest_vi_ablation_fresh
Files: ['KAGGLE_REST_VI.md', 'README.md', 'absa_pipeline.py', 'dataset', 'kaggle_rest_vi_ablation_fresh.ipynb', 'requirements.txt', 'train.py', 'tune.py']


## 2. Cài đặt thư viện

Kaggle cần bật Internet để tải multilingual DeBERTa lần đầu.

In [3]:
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 43.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], returncode=0)

## 3. Kiểm tra package và dataset tiếng Việt

Cell xác nhận số câu, số aspect, nhãn và vị trí `term/from/to`. Các trường dependency syntax không được yêu cầu.

In [4]:
os.chdir(PROJECT)
required = [
    Path('train.py'), Path('tune.py'), Path('absa_pipeline.py'),
    Path('dataset/rest_vi/train.multiple.json'),
    Path('dataset/rest_vi/test.multiple.json'),
    Path('dataset/rest_vi/aspects.json'),
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Thiếu file bắt buộc: {missing}'

forbidden_suffixes = {'.pt', '.pth', '.bin', '.safetensors'}
history_files = [str(p.relative_to(PROJECT)) for p in PROJECT.rglob('*') if p.suffix in forbidden_suffixes]
assert not history_files, f'Package chứa checkpoint cũ: {history_files[:20]}'

from collections import Counter
for split in ['train', 'test']:
    rows = json.loads(Path(f'dataset/rest_vi/{split}.multiple.json').read_text(encoding='utf-8'))
    aspects = [a for row in rows for a in row.get('aspects', [])]
    errors = []
    for row in rows:
        assert row['sentence'] == ' '.join(row['tokens'])
        for aspect in row['aspects']:
            actual = row['tokens'][aspect['from']:aspect['to']]
            if actual != aspect['term']:
                errors.append((row['index'], aspect['term'], actual))
    assert not errors, errors[:10]
    print(split, 'sentences=', len(rows), 'aspect_samples=', len(aspects),
          'labels=', dict(Counter(a['polarity'] for a in aspects)))

subprocess.run([sys.executable, 'tune.py', '--help'], check=True)

train sentences= 1980 aspect_samples= 3608 labels= {'negative': 807, 'positive': 2164, 'neutral': 637}
test sentences= 600 aspect_samples= 1120 labels= {'positive': 728, 'neutral': 196, 'negative': 196}
usage: tune.py [-h] [--dataset {lap,rest,rest_vi,twi}] [--model MODEL]
               [--epochs EPOCHS] [--final-epochs FINAL_EPOCHS]
               [--patience PATIENCE] [--batch-size BATCH_SIZE]
               [--gradient-accumulation-steps GRADIENT_ACCUMULATION_STEPS]
               [--weight-decay WEIGHT_DECAY] [--valid-ratio VALID_RATIO]
               [--split-seed SPLIT_SEED] [--tuning-seeds TUNING_SEEDS]
               [--validation-seed-pairs VALIDATION_SEED_PAIRS]
               [--final-seeds FINAL_SEEDS] [--output-dir OUTPUT_DIR]
               [--final-output-dir FINAL_OUTPUT_DIR] [--amp]
               [--gradient-checkpointing] [--run-final]
               [--keep-trial-checkpoints] [--resume]
               [--search-space {ablation_all,ablation_core,ablation_desc,initia

CompletedProcess(args=['/usr/bin/python3', 'tune.py', '--help'], returncode=0)

## 4. Chạy core ablation

Mặc định chạy 18 validation run và 3 final run. Với GPU 16 GB, nếu hết bộ nhớ hãy đổi `BATCH_SIZE=4`, `GRAD_ACCUM=4`; effective batch size vẫn bằng 16.

Để kiểm tra nhanh pipeline trước, đặt `QUICK_CHECK=True`. Chế độ này chỉ chạy một cấu hình/một seed và không phải kết quả báo cáo cuối.

In [5]:
os.chdir(PROJECT)

MODEL = 'microsoft/mdeberta-v3-base'
VALID_RATIO = 0.15
BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 12
FINAL_EPOCHS = 15
SEED_PAIRS = '2024:2024,42:42,3407:3407'
FINAL_SEEDS = '2024,42,3407'
QUICK_CHECK = False

cmd = [
    sys.executable, 'tune.py',
    '--dataset', 'rest_vi',
    '--model', MODEL,
    '--search-space', 'ablation_core',
    '--validation-seed-pairs', '2024:2024' if QUICK_CHECK else SEED_PAIRS,
    '--valid-ratio', str(VALID_RATIO),
    '--final-seeds', '2024' if QUICK_CHECK else FINAL_SEEDS,
    '--epochs', '2' if QUICK_CHECK else str(EPOCHS),
    '--final-epochs', '2' if QUICK_CHECK else str(FINAL_EPOCHS),
    '--patience', '2' if QUICK_CHECK else '3',
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRAD_ACCUM),
    '--amp', '--gradient-checkpointing', '--run-final',
    '--output-dir', str(CORE_TUNING),
    '--final-output-dir', str(CORE_FINAL),
]
if QUICK_CHECK:
    cmd.extend(['--trials', 'aspect_attention_term'])

print(' '.join(cmd))
subprocess.run(cmd, check=True)

/usr/bin/python3 tune.py --dataset rest_vi --model microsoft/mdeberta-v3-base --search-space ablation_core --validation-seed-pairs 2024:2024,42:42,3407:3407 --valid-ratio 0.15 --final-seeds 2024,42,3407 --epochs 12 --final-epochs 15 --patience 3 --batch-size 8 --gradient-accumulation-steps 2 --amp --gradient-checkpointing --run-final --output-dir /kaggle/working/tuning_rest_vi_ablation_core --final-output-dir /kaggle/working/outputs_rest_vi_ablation_core
RUN: /usr/bin/python3 train.py --dataset rest_vi --model microsoft/mdeberta-v3-base --epochs 12 --patience 3 --batch-size 8 --gradient-accumulation-steps 2 --weight-decay 0.01 --valid-ratio 0.15 --split-seed 2024 --seed 2024 --output-dir /kaggle/working/tuning_rest_vi_ablation_core --run-name tune_baseline_cls_split2024_seed2024 --evaluation-split validation --pooling-strategy cls --input-format term --encoder-layer-pooling last --encoder-learning-rate 8e-06 --head-learning-rate 2e-05 --dropout 0.3 --class-weighting none --neutral-weig

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-22 19:45:47.541614: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782157547.704924      63 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782157547.750340      63 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 3064,
      "labels": {
        "negative": 684,
        "neutral": 541,
        "positive": 1839
      },
      "unique_aspects": 1079
    },
    "train_all": {
      "samples": 3606,
      "labels": {
        "negative": 807,
        "neutral": 637,
        "positive": 2162
      },
      "unique_aspects": 1223,
      "unseen_aspect_samples": 148,
      "unseen_aspect_ratio": 0.041
    },
    "valid": {
      "samples": 542,
      "labels": {
        "negative": 123,
        "neutral": 96,
        "positive": 323
      },
      "unique_aspects": 286,
      "unseen_aspect_samples": 148,
      "unseen_aspect_ratio": 0.2731
    },
    "test": {
      "samples": 1120,
      "labels": {
        "negative": 196,
        "neutral": 196,
        "positive": 728
      },
      "unique_aspects": 524,
      "unseen_aspect_samples": 362,
      "unseen_aspect_ratio": 0.3232
    },
  

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-22 20:06:45.300399: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782158805.319103     108 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782158805.325084     108 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 3064,
      "labels": {
        "negative": 683,
        "neutral": 542,
        "positive": 1839
      },
      "unique_aspects": 1070
    },
    "train_all": {
      "samples": 3606,
      "labels": {
        "negative": 807,
        "neutral": 637,
        "positive": 2162
      },
      "unique_aspects": 1223,
      "unseen_aspect_samples": 155,
      "unseen_aspect_ratio": 0.043
    },
    "valid": {
      "samples": 542,
      "labels": {
        "negative": 124,
        "neutral": 95,
        "positive": 323
      },
      "unique_aspects": 287,
      "unseen_aspect_samples": 155,
      "unseen_aspect_ratio": 0.286
    },
    "test": {
      "samples": 1120,
      "labels": {
        "negative": 196,
        "neutral": 196,
        "positive": 728
      },
      "unique_aspects": 524,
      "unseen_aspect_samples": 362,
      "unseen_aspect_ratio": 0.3232
    },
   

...

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-23 01:10:24.868050: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782177024.889420     595 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782177024.896342     595 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 3064,
      "labels": {
        "negative": 684,
        "neutral": 541,
        "positive": 1839
      },
      "unique_aspects": 1079
    },
    "train_all": {
      "samples": 3606,
      "labels": {
        "negative": 807,
        "neutral": 637,
        "positive": 2162
      },
      "unique_aspects": 1223,
      "unseen_aspect_samples": 148,
      "unseen_aspect_ratio": 0.041
    },
    "valid": {
      "samples": 542,
      "labels": {
        "negative": 123,
        "neutral": 96,
        "positive": 323
      },
      "unique_aspects": 286,
      "unseen_aspect_samples": 148,
      "unseen_aspect_ratio": 0.2731
    },
    "test": {
      "samples": 1120,
      "labels": {
        "negative": 196,
        "neutral": 196,
        "positive": 728
      },
      "unique_aspects": 524,
      "unseen_aspect_samples": 362,
      "unseen_aspect_ratio": 0.3232
    },
  

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-23 01:36:29.319145: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782178589.339033     623 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782178589.345762     623 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 3064,
      "labels": {
        "negative": 684,
        "neutral": 541,
        "positive": 1839
      },
      "unique_aspects": 1079
    },
    "train_all": {
      "samples": 3606,
      "labels": {
        "negative": 807,
        "neutral": 637,
        "positive": 2162
      },
      "unique_aspects": 1223,
      "unseen_aspect_samples": 148,
      "unseen_aspect_ratio": 0.041
    },
    "valid": {
      "samples": 542,
      "labels": {
        "negative": 123,
        "neutral": 96,
        "positive": 323
      },
      "unique_aspects": 286,
      "unseen_aspect_samples": 148,
      "unseen_aspect_ratio": 0.2731
    },
    "test": {
      "samples": 1120,
      "labels": {
        "negative": 196,
        "neutral": 196,
        "positive": 728
      },
      "unique_aspects": 524,
      "unseen_aspect_samples": 362,
      "unseen_aspect_ratio": 0.3232
    },
  

CompletedProcess(args=['/usr/bin/python3', 'tune.py', '--dataset', 'rest_vi', '--model', 'microsoft/mdeberta-v3-base', '--search-space', 'ablation_core', '--validation-seed-pairs', '2024:2024,42:42,3407:3407', '--valid-ratio', '0.15', '--final-seeds', '2024,42,3407', '--epochs', '12', '--final-epochs', '15', '--patience', '3', '--batch-size', '8', '--gradient-accumulation-steps', '2', '--amp', '--gradient-checkpointing', '--run-final', '--output-dir', '/kaggle/working/tuning_rest_vi_ablation_core', '--final-output-dir', '/kaggle/working/outputs_rest_vi_ablation_core'], returncode=0)

## 5. Tổng hợp kết quả

Bảng đầu là kết quả validation ablation; bảng sau là kết quả official test của cấu hình thắng.

In [6]:
import pandas as pd

summary_path = CORE_TUNING / 'tuning_summary.csv'
final_path = CORE_FINAL / 'final_runs_summary.csv'
meanstd_path = CORE_FINAL / 'final_metrics_mean_std.json'
selection_path = CORE_TUNING / 'selected_configuration.json'

summary = pd.read_csv(summary_path)
cols = [c for c in ['trial','mean_macro_f1','std_macro_f1','robust_macro_f1','mean_accuracy','mean_ece','pooling_strategy','neutral_weight_multiplier'] if c in summary.columns]
display(summary[cols])

if selection_path.exists():
    print('Selected configuration:')
    print(json.dumps(json.loads(selection_path.read_text()), indent=2))
if final_path.exists():
    display(pd.read_csv(final_path))
if meanstd_path.exists():
    print('Final mean/std:')
    print(json.dumps(json.loads(meanstd_path.read_text()), indent=2))

,trial,mean_macro_f1,std_macro_f1,robust_macro_f1,mean_accuracy,mean_ece,pooling_strategy,neutral_weight_multiplier
0,hybrid_term,0.694633,0.001676,0.692957,0.778933,0.048167,hybrid,1.00
1,baseline_cls,0.681933,0.005967,0.675967,0.759867,0.069067,cls,1.00
2,aspect_attention_neutral125,0.700567,0.024837,0.675729,0.772167,0.059433,aspect_attention,1.25
3,cls_neutral125,0.680067,0.004358,0.675708,0.767833,0.061567,cls,1.25
4,aspect_attention_term,0.691767,0.016481,0.675285,0.770333,0.050000,aspect_attention,1.00
5,cls_neutral150,0.669200,0.002707,0.666493,0.753700,0.068300,cls,1.50


Selected configuration:
{
  "selection_metric": "mean Macro-F1 minus one standard deviation",
  "search_space": "ablation_core",
  "final_split_seed": 2024,
  "validation_seed_pairs": [
    [
      2024,
      2024
    ],
    [
      42,
      42
    ],
    [
      3407,
      3407
    ]
  ],
  "best_configuration": {
    "name": "hybrid_term",
    "pooling_strategy": "hybrid",
    "input_format": "term",
    "encoder_layer_pooling": "last",
    "encoder_learning_rate": 8e-06,
    "head_learning_rate": 2e-05,
    "dropout": 0.3,
    "class_weighting": "none",
    "neutral_weight_multiplier": 1.0,
    "label_smoothing": 0.02,
    "loss_type": "cross_entropy",
    "scheduler": "linear",
    "warmup_ratio": 0.1,
    "max_length": 128
  },
  "best_validation_summary": {
    "trial": "hybrid_term",
    "mean_macro_f1": 0.694633,
    "std_macro_f1": 0.001676,
    "robust_macro_f1": 0.692957,
    "mean_accuracy": 0.778933,
    "mean_loss": 0.765267,
    "mean_log_loss": 0.5729,
    "mean_ece"

,seed,best_epoch,accuracy,precision_macro,recall_macro,macro_f1,weighted_f1,roc_auc_macro,log_loss,expected_calibration_error,temperature
0,2024,2,0.7964,0.7599,0.6421,0.6813,0.7789,0.9048,0.5372,0.0582,0.982818
1,42,6,0.8366,0.7751,0.7360,0.7463,0.8265,0.9283,0.4655,0.0381,2.014446
2,3407,3,0.8080,0.7364,0.6915,0.6886,0.7883,0.9152,0.4865,0.0237,1.454474


Final mean/std:
{
  "accuracy": {
    "mean": 0.813667,
    "std": 0.016894
  },
  "precision_macro": {
    "mean": 0.757133,
    "std": 0.01592
  },
  "recall_macro": {
    "mean": 0.689867,
    "std": 0.038352
  },
  "macro_f1": {
    "mean": 0.7054,
    "std": 0.029074
  },
  "weighted_f1": {
    "mean": 0.7979,
    "std": 0.020584
  },
  "roc_auc_macro": {
    "mean": 0.9161,
    "std": 0.009615
  },
  "log_loss": {
    "mean": 0.4964,
    "std": 0.030097
  },
  "expected_calibration_error": {
    "mean": 0.04,
    "std": 0.014148
  }
}


## 6. Ghi chú về aspect description

`aspects.json` của bản dịch chủ yếu lưu thống kê và term nguồn; trường description đang rỗng. Vì vậy notebook chỉ chạy core ablation với raw aspect term. Không chạy `ablation_desc` cho đến khi có mô tả tiếng Việt được kiểm tra.

In [7]:
aspect_info = json.loads(Path('dataset/rest_vi/aspects.json').read_text(encoding='utf-8'))
non_empty_descriptions = sum(bool(item.get('description', '').strip()) for item in aspect_info.values())
print('Unique translated aspects:', len(aspect_info))
print('Non-empty Vietnamese descriptions:', non_empty_descriptions)
assert non_empty_descriptions == 0, 'Nếu đã bổ sung description, có thể chạy ablation_desc riêng.'

Unique translated aspects: 1562
Non-empty Vietnamese descriptions: 0


## 7. Đóng gói kết quả

Checkpoint và ZIP artifact lồng nhau bị loại mặc định để file kết quả nhỏ hơn. JSON, CSV, prediction và biểu đồ vẫn được giữ.

In [8]:
INCLUDE_CHECKPOINTS = False
folders = [CORE_TUNING, CORE_FINAL]

with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in folders:
        if not folder.exists():
            continue
        for path in folder.rglob('*'):
            if not path.is_file():
                continue
            if not INCLUDE_CHECKPOINTS and path.suffix in {'.pt', '.pth', '.bin', '.safetensors'}:
                continue
            if path.name.endswith('_artifacts.zip'):
                continue
            zf.write(path, path.relative_to(WORK))

print('Created:', RESULT_ZIP)
print('Size MB:', round(RESULT_ZIP.stat().st_size / 1024 / 1024, 2))

Created: /kaggle/working/absa_rest_vi_results_fresh.zip
Size MB: 9.3
